In [1]:
import numpy 
from matplotlib import pyplot
%matplotlib inline 

In [2]:
pyplot.rcParams['font.family'] = 'serif'
pyplot.rcParams['font.size'] = 16

In [47]:
################ Functions to calculate the UQ's ###################
# u_input
def inputUQ(vel_1, vel_2, du):
    dO = numpy.abs(vel_1 -vel_2)  # Diff in Outputs 
#     du = numpy.abs()         # Diff in uncertainties
    u_input  = dO/du              # Sensitivity based on uncertainty
    #print(dO)
    return u_input

# u_num  - Functions for calculation
def fixPtItr(r_21,r_32,e_21,e_32,s,p):
    p_new = (1/numpy.log(r_21))*numpy.abs(numpy.log(numpy.abs(e_32/e_21)
                                               )+numpy.log(((r_21**p) - s)/
                                                    (((r_32**p) - s))))
    return p_new

# u_d

#Convert to m/s
def convVS(num_LM): 
    num_VS = num_LM *(1/60000)/(numpy.pi*(4.54/2000)**2)
    return num_VS

## Calcualte the CGI for u_num

In [21]:
maxiter=101
rtol=1e-20
N = 100
fs = 1.25

# Mesh or grid size h 
h1 = 0.000122  # Fine Mesh (M 4-3)
h2 = 0.000190  # Medium Mesh (M 3-4)
h3 = 0.000253  # Coarse Mesh (M 3-0)

# Phi = Velocity @ 1mm above mouthpiece Max
phi_1_raw = 0.982088387    # velocity in m/s
phi_2_raw = 0.9809986949   # velocity in m/s
phi_3_raw = 0.9636819363   # velocity in m/s

# Mesh: h_coarse/h_fine 
r_21 =  h2/h1 # h_2/h_1   
r_32 =  h3/h2 # h_3/h_2
e_21 = phi_2-phi_1 
e_32 = phi_3-phi_2 

# s is the sign of e32/e21
if (e_32/e_21)< 0:
    s = -1
elif (e_32/e_21)> 0:
    s = 1    
    
conv = []          # convergence history
diff = rtol + 1    # initial difference
ite = 0            # iteration index
p = numpy.zeros(N) # Holds all p values 

# Initial p value assuming q(p) = 0 
p[0] = (1/numpy.log(r_21))*numpy.abs(numpy.log(numpy.abs(e_32/e_21)))

n = 0;  # Since at p[0] is initial value 
while diff > rtol and ite <maxiter: 
    # Calculating the next p value 
    p[n+1] = fixPtItr(r_21,r_32,e_21,e_32,s,p[n])

    # Compute the relative L2-Norm of the difference
    diff = numpy.sqrt(numpy.sum((p[n+1] - p[n])**2))
    #diff = numpy.abs(p[n+1] - p[n])

    #conv.append(diff)
    ite += 1
    n += 1
ord_p = p[n]

# Calculate the Extrapolated Values
phi_ext_21 = ((phi_1*r_21**ord_p)-phi_2)/(r_21**ord_p-1) # Extrapolated values 
phi_ext_32 = ((phi_1*r_32**ord_p)-phi_2)/(r_32**ord_p-1) # Extrapolated values 

# Approximate Relative Error as % (*100)
e_a_32 = numpy.abs((phi_2-phi_3)/phi_2)*100
e_a_21 = numpy.abs((phi_1-phi_2)/phi_1)*100

# Extrapolated Relative Error
e_ext_21 = numpy.abs((phi_ext_21-phi_1)/phi_ext_21)*100

# Grid Convergence Index  (no dimension?)
gci_fine_21 = (fs*e_a_21)/((r_21**ord_p)-1)

In [35]:
# GCI to u_num 
u_num = gci_fine_21/2 

## Calculate u_input (m/s)

In [60]:
# u_input(vel_1, vel_2, u1, u2)
#u1 =  # Uncertainty Associated w/ VFR for Sim 1 
#u2 =  # Uncertainty Associated w/ VFR for Sim 2

#du_Lm = 0.001 # VFR inlet L/min
du_Lm = 0.01 # VFR inlet L/min 

du = convVS(du_Lm)
# Convert to m/s
du = du_Lm *(1/60000)/(numpy.pi*(4.54/2000)**2)

u_input = inputUQ(phi_1, phi_2, du)  # dV/ds : DIff in velocity output /diff in inlet VFR 
#u_input = numpy.abs(phi_1- phi_2)     # Only diff in output 

In [59]:
du

0.0010295493352733111

In [61]:
u_input

0.10584165932278579

## Calculate u_d  (m/s)

In [52]:
# Measured variables
s_u = 0.0     # Random Uncertainty 
b_u_LM = 0.001   # Systematic Uncerainty (L/min)

b_u = convVS(b_u_LM) # Convert to m/s

u_d = numpy.sqrt(s_u**2 + b_u**2)  # L/min

In [54]:
b_u

0.0010295493352733111

# Calculate u_val (m/s)

In [62]:
u_val = numpy.sqrt(u_d**2 + u_input**2 + u_num**2)

In [63]:
u_val

0.10585051753160279

In [64]:
u_d

0.0010295493352733111

In [65]:
u_num

0.00090290734504779903